In [1]:
import pandas as pd
import sys
import os

sys.path.append('/home1/gvanerven/code/lailab')
from models.classes_pydantic import RegistroPedido, ResumoPedido, ResumoPedidoSimples

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from tqdm import tqdm
import json
import logging


In [2]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

model_id = "Qwen/Qwen3-8B"

sel_cols = ['IdPedido', 
            'Ano',
            'ProtocoloPedido', 
            'Orgaodestinatario', 
            'ResumoSolicitacao', 
            'DetalhamentoSolicitacao', 
            'AssuntoPedido', 
            'SubAssuntoPedido', 
            'Tag', 
            'Resposta', 
            'Decisao',
            'DetalhamentoDecisao',
            'MotivoNegativaAcesso']

In [3]:
ANO = '2026'

In [4]:
df = pd.read_parquet('/home1/gvanerven/code/lailab/etl/datasets/pedidos_lai.parquet', columns=sel_cols, filters=[('Ano', '==', ANO)])
print(f"DF Shape: {df.shape}")

DF Shape: (13722, 13)


In [5]:
pedidos = []
for _, row in df.iterrows():
    pedidos.append(RegistroPedido(**row.to_dict()))
    
assert df.shape[0] == len(pedidos)

In [6]:
system_prompt = f"""
Você é um Analista de Pedidos de Acesso à Informação e deve realizar tarefas de consolidação de informações de um pedido de acesso à informação.

O pedido de acesso à informação, ou simplesmente pedido, é realizada a partir dos seguintes campos com as respectivas descrições sobre o que tratam:
    "IdPedido": Número inteiro identificando unicamente o pedido.
    "ProtocoloPedido": Número do protocolo do pedido, com 17 caracteres numéricos.
    "Orgaodestinatario": Nome do órgão do governo de destino do pedidos.
    "ResumoSolicitacao": Resumo do pedido, que pode ter um valor ou não.
    "DetalhamentoSolicitacao": O texto principal do pedido de acesso à informação.
    "AssuntoPedido": O assunto em geral do pedido selecionado de uma lista finita de opções.
    "SubAssuntoPedido": O subassunto em geral do pedido selecionado de uma lista finita de opções.
    "Tag": Palavras-chave gerais para o pedido.
    "Resposta": A resposta do órgão para o pedido.
    "Decisao": A decisão em geral do órgão ao pedido selecionado de uma lista finita de opções.
    "DetalhamentoDecisao": Informações adicionais sobre a decisão do órgão para o pedido, que pode ter um valor ou não.
    "MotivoNegativaAcesso": Motivação sobre a decisão do órgão em caso de negativa de acesso ao pedido, que pode ter um valor ou não.

O formato json do pedido possui o seguinte esquema:

{RegistroPedido.model_json_schema()}

Para o pedido, deve-se extrair as seguintes informações:
    "IdPedido": Id do Pedido analisado.
    "resumo": Corrija eventuais erros de escrita e escreva um resumo em linguagem formal de um parágrafo no máximo sobre o pedido contendo as informações mais relevantes como, por exemplo: O quê? (What): O fato, o acontecimento central; Quem? (Who): Os agentes, sujeitos envolvidos; Quando? (When): O tempo, a data ou momento do ocorrido; Onde? (Where): O local, o espaço físico onde o fato ocorreu; Como? (How): O modo, as circunstâncias em que o fato se desenrolou; Por quê? (Why): O motivo, a razão ou a causa do fato. Inlcua o número do protocolo do pedido no texto.

Extraia as informações do pedido de acesso à informação do usuário delimitado pelas tags <pedido></pedido>:

"""


In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
).eval()


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
batch_size = 32
resumos_final = []
for i in tqdm(range(0, len(pedidos), batch_size)):
    resumos = []
    batch = []
    for pedido in pedidos[i:i+batch_size]:
        user_prompt = f"""
            <pedido>
                {json.dumps(pedido.model_dump_json(), indent=2)}
            </pedido>
            
            Retorne o resultado contento apenas os campos do formato json abaixo:
                {ResumoPedidoSimples.model_json_schema()}
        """
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            temperature = 0.01,
            enable_thinking=False
        )
        batch.append(text)
    
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, padding_side='left').to(model.device)

    generated_ids = model.generate(
        **inputs,
        use_cache=True,
        max_new_tokens=1024
    )

    for gen_id in generated_ids:
        output_ids = gen_id[len(inputs.input_ids[0]):].tolist()
        content = tokenizer.decode(output_ids, skip_special_tokens=True)
        try:
            aux = ResumoPedidoSimples(**json.loads(content)).model_dump()
            resumos.append(aux)
            resumos_final.append(aux)

        except Exception as e:
            logger.error(f'error procesing content: {content}. ERROR: {e}')

    tmp_df = pd.DataFrame(resumos)
    tmp_df.to_parquet(f"/home1/gvanerven/code/lailab/resumos/pedidos_resumos_tmp_df_batch{i}-{i+batch_size}_{ANO}.parquet", index=False)
    break
# mi2104x - batch 48 18:47 min
# mi2104x - batch 48 04:16 min
# mi2104x - batch  8 01:56 min
# mi2104x - batch  8 31 s
# mi2104x - batch  32 1:58 s
# mi2104x - batch  4 01:12 min
# mi2508x - batch 48 18:47 min
# mi2508x - batch 32 09:49 min
# mi2508x - batch  8 01:46 min

  0%|          | 0/286 [04:16<?, ?it/s]


In [9]:
final_df = pd.DataFrame(resumos_final)
final_df.to_parquet(f"/home1/gvanerven/code/lailab/resumos/pedidos_resumos_final_{ANO}.parquet", index=False)